<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB04_cytochrome_alignment_conservation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB04 — Cytochrome alignment and conservation
**MICR4203 / MICR5203 · Fall 2026 · Version 1.0**

Save a copy in Drive before starting. Run cells from top to bottom, then complete the report questions. This notebook combines the pairwise-alignment and guide-tree lesson from NB06 with a completed conservation-analysis workflow for NB07.

**Biological question:** Which cytochrome alignment positions are conserved, and how reliable is that conclusion?

You will compare sequences, interpret a guide tree, examine an instructor-curated multiple-sequence alignment (MSA), and compare three column scores and a sequence logo. Figures export automatically as **300-dpi PNG**, plus SVG and PDF. All positions are **1-based alignment columns**, unless labeled as reference-residue positions.

**Data scope:** The original course panel contains seven **synthetic cytochrome-c-inspired teaching homologs**, not measured species observations. We inspect the curated MSA; the pairwise alignments and tree calculated here do **not** generate that MSA. An optional MAFFT extension below generates a new alignment.

**Learning goals:** explain N(N−1)/2 comparisons and D=1−I; distinguish a guide tree from an evolutionary inference; identify conserved, variable, and gap-rich columns; interpret residue colors and sequence logos; prepare figures with evidence-based captions.


## 1. Install tools and locate the course folders
Run once per fresh Colab session. A local Jupyter session also works; it uses a course folder in the current directory. Existing input files are preserved. Each run gets its own output folder.


In [ ]:
%pip install -q biopython matplotlib pandas numpy logomaker
from pathlib import Path
from datetime import datetime
from collections import Counter
import hashlib, json, re, shutil, subprocess, sys, importlib.metadata
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import logomaker
from Bio import Align, AlignIO, Phylo, SeqIO
from Bio.Align import substitution_matrices
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor
from IPython.display import display

NOTEBOOK_ID = 'NB04_cytochrome_alignment_conservation'
NOTEBOOK_VERSION = '1.0'
COURSE_FOLDER_NAME = 'BIOINFO4-5203-F26'
try:
    from google.colab import drive
except ImportError:
    COURSE_DIR = Path.cwd() / COURSE_FOLDER_NAME
else:
    drive.mount('/content/drive')
    candidates = [Path('/content/drive/MyDrive') / COURSE_FOLDER_NAME,
                  Path('/content/drive/MyDrive/Teaching') / COURSE_FOLDER_NAME]
    COURSE_DIR = next((p for p in candidates if p.exists()), candidates[0])
DATA_DIR = COURSE_DIR / 'Data' / NOTEBOOK_ID
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_DIR = COURSE_DIR / 'Outputs' / NOTEBOOK_ID / RUN_ID
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Inputs:', DATA_DIR)
print('This run:', OUTPUT_DIR)


## 2. Settings and report-image export
Keep these settings for your first run. Later, change the logo window or gap threshold and explain the effect. **GAP_THRESHOLD is an inclusive caution threshold, not a statistical significance cutoff.**


In [ ]:
GAP_OPEN = -10.0
GAP_EXTEND = -0.5
GAP_THRESHOLD = 0.10
BLOCK_WIDTH = 50
LOGO_START = 1
LOGO_END = 40
FORMATS = ('png', 'svg', 'pdf')
AA = list('ACDEFGHIKLMNPQRSTVWY')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 14,
                     'svg.fonttype': 'none', 'pdf.fonttype': 42})
figure_files = []
def save_figure(fig, stem):
    for extension in FORMATS:
        path = OUTPUT_DIR / f'{stem}.{extension}'
        fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
        if path not in figure_files:
            figure_files.append(path)
    plt.show()
    plt.close(fig)
    print('Saved:', stem + '.png (+ SVG and PDF)')


## 3. Obtain and validate the same input panel used in NB06
We retain the original NB06 data location on the course repository; renumbering this notebook does not imply that the remote data were moved. Files already in NB04's Data folder take precedence, followed by files in the old NB06 folder, then a repository download. If download is unavailable, obtain the **two original FASTA files** from the instructor and place them in the printed input folder. No substitute sequences are silently created.

For a different instructor-approved panel, replace both files consistently. Validation checks unique IDs, supported amino acids, and exact agreement between ungapped aligned sequences and input sequences.


In [ ]:
UNALIGNED_PATH = DATA_DIR / 'cytochrome_c_teaching_unaligned.fasta'
ALIGNED_PATH = DATA_DIR / 'cytochrome_c_teaching_reference_alignment.fasta'
DATA_BASE_URL = ('https://raw.githubusercontent.com/RobBurnap/'
                 'Bioinformatics-MICR4203-MICR5203/main/data/NB06_msa_generation_qc/')
input_sources = {}
for path in (UNALIGNED_PATH, ALIGNED_PATH):
    old = COURSE_DIR / 'Data' / 'NB06_msa_generation_qc' / path.name
    if path.exists():
        input_sources[path.name] = 'Existing NB04 input'
    elif old.exists():
        shutil.copy2(old, path)
        input_sources[path.name] = str(old)
    else:
        temporary = path.with_suffix('.download')
        try:
            urlretrieve(DATA_BASE_URL + path.name, temporary)
            temporary.replace(path)
            input_sources[path.name] = DATA_BASE_URL + path.name
        except Exception as error:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f'Obtain {path.name} from your instructor and put it in {DATA_DIR}, then rerun.') from error

records = list(SeqIO.parse(UNALIGNED_PATH, 'fasta'))
msa = AlignIO.read(ALIGNED_PATH, 'fasta')
names = [r.id for r in records]
sequences = [str(r.seq).upper() for r in records]
N = len(records)
assert N >= 2, 'At least two sequences are required.'
assert len(set(names)) == N, 'Duplicate input IDs.'
assert all(s and set(s) <= set(AA) for s in sequences), 'Input must contain the 20 standard amino acids only.'
assert len(msa) == N and len({r.id for r in msa}) == N, 'Missing or duplicate aligned IDs.'
aligned_by_id = {r.id: str(r.seq).upper() for r in msa}
assert set(aligned_by_id) == set(names), 'Aligned and unaligned IDs differ.'
for name, sequence in zip(names, sequences):
    assert set(aligned_by_id[name]) <= set(AA + ['-']), f'Unsupported aligned symbol: {name}'
    assert aligned_by_id[name].replace('-', '') == sequence, f'Sequence mismatch: {name}'
arr = np.array([list(aligned_by_id[name]) for name in names])
L = arr.shape[1]
assert L > 0
expected_pairs = N * (N - 1) // 2
print(f'{N} sequences; {expected_pairs} unique comparisons; {L} alignment columns')
if N != 7:
    print('Note: this is not the original seven-sequence panel size.')
display(pd.DataFrame({'sequence': names, 'length_aa': list(map(len, sequences))}))


## 4. Pairwise identities and distances
Global pairwise alignment uses BLOSUM62 with affine gaps. Identity here is **identical residue–residue pairs / all residue–residue pairs**; columns containing a gap are excluded. Consequently, high identity can coexist with incomplete overlap. We also save the number of paired residues and alignment-column coverage.

**Predict:** How many comparisons are required for seven sequences? Why divide by two?


In [ ]:
pair_aligner = Align.PairwiseAligner()
pair_aligner.mode = 'global'
pair_aligner.substitution_matrix = substitution_matrices.load('BLOSUM62')
pair_aligner.open_gap_score = GAP_OPEN
pair_aligner.extend_gap_score = GAP_EXTEND
identity = pd.DataFrame(np.eye(N), index=names, columns=names)
pair_rows = []
for i in range(N):
    for j in range(i + 1, N):
        alignment = pair_aligner.align(sequences[i], sequences[j])[0]
        indices = alignment.indices
        paired = [(a, b) for a, b in zip(*indices) if a >= 0 and b >= 0]
        if not paired:
            raise ValueError('No residue pairs; identity is undefined.')
        value = sum(sequences[i][a] == sequences[j][b] for a, b in paired) / len(paired)
        identity.iloc[i, j] = identity.iloc[j, i] = value
        pair_rows.append(dict(sequence_1=names[i], sequence_2=names[j], identity=value,
                              distance=1-value, alignment_score=alignment.score,
                              paired_residues=len(paired),
                              paired_column_fraction=len(paired)/indices.shape[1]))
pairwise_results = pd.DataFrame(pair_rows)
assert len(pairwise_results) == expected_pairs
distance = 1 - identity
display(pairwise_results.sort_values('distance').round(3))
fig, axes = plt.subplots(1, 2, figsize=(15, 6), layout='constrained')
for ax, table, title, cmap in [(axes[0], identity, 'Pairwise identity', 'Blues'),
                               (axes[1], distance, 'Distance = 1 − identity', 'Oranges')]:
    im = ax.imshow(table, vmin=0, vmax=1, cmap=cmap)
    ax.set_xticks(range(N), names, rotation=65, ha='right')
    ax.set_yticks(range(N), names)
    for i in range(N):
        for j in range(N):
            v = table.iloc[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', color='white' if v > .6 else 'black', fontsize=9)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=.7, label='Fraction (0–1)')
save_figure(fig, 'fig01_pairwise_identity_distance')


## 5. Guide-tree reasoning
D = 1−I is an introductory distance with no correction for repeated substitutions. The neighbor-joining tree below visualizes those distances. **It was not used to build the curated MSA.** Its drawing orientation is not an inferred evolutionary root; negative branch estimates can occur with these simple distances.

Classical progressive alignment joins sequences/profiles following a guide tree, and early gap decisions can persist. The closest pair is a plausible starting pair, but does not uniquely determine every algorithm's alignment order.


In [ ]:
triangle = [[float(distance.iloc[i, j]) for j in range(i + 1)] for i in range(N)]
guide_tree = DistanceTreeConstructor().nj(DistanceMatrix(names, triangle))
fig, ax = plt.subplots(figsize=(11, 6), layout='constrained')
Phylo.draw(guide_tree, axes=ax, do_show=False,
           label_func=lambda clade: clade.name if clade.is_terminal() else None)
ax.set_title('Distance-based guide-tree illustration — no inferred root')
ax.set_xlabel('Branch distance (1 − pairwise identity)')
save_figure(fig, 'fig02_guide_tree')
Phylo.write(guide_tree, str(OUTPUT_DIR / 'guide_tree.nwk'), 'newick')
print('Closest pair(s):')
display(pairwise_results.loc[np.isclose(pairwise_results.distance, pairwise_results.distance.min())])
print('Mean distance to other sequences (a descriptive outlier measure):')
display((distance.sum(axis=1)/(N-1)).sort_values(ascending=False).rename('mean_distance'))


## 6. Three views of conservation, with gap coverage
For each column, let n be the non-gap count and p(a) the frequency of residue a among those n residues.

- **Identity conservation:** max p(a). A column with a single residue and six gaps scores 1.0.
- **Entropy conservation:** 1 − H/log₂(20), where H = −Σ p(a)log₂p(a). A fixed 20-residue scale makes comparisons consistent, but seven sequences cannot sample 20 residue types. This is a descriptive score, not a significance test.
- **Occupancy-weighted identity:** (n/N) × max p(a), equivalent to the most common residue count divided by N. This deliberately penalizes missing coverage; it is not a substitute for a model of evolution.

An all-gap column has undefined identity/entropy (NaN) and weighted identity 0. Tied consensus residues are shown explicitly; the displayed consensus chooses the alphabetically first tied residue. Scores are unweighted: closely related or duplicated sequences can bias them.


In [ ]:
rows, probability_rows = [], []
reference_name = names[0]
reference_position = 0
for j in range(L):
    counts = Counter(arr[:, j][arr[:, j] != '-'])
    n = sum(counts.values())
    p = np.array([counts.get(a, 0)/n if n else 0 for a in AA])
    nonzero = p[p > 0]
    entropy = float(-np.sum(nonzero*np.log2(nonzero))) if n else np.nan
    top = max(counts.values()) if n else 0
    tied = sorted(a for a, count in counts.items() if count == top)
    if arr[0, j] != '-':
        reference_position += 1
    rows.append(dict(column_1_based=j+1, reference_id=reference_name,
                     reference_position=reference_position if arr[0, j] != '-' else None,
                     consensus=tied[0] if tied else '-', consensus_ties='/'.join(tied),
                     non_gap_count=n, occupancy=n/N, gap_fraction=1-n/N,
                     identity_conservation=top/n if n else np.nan,
                     entropy_bits=entropy,
                     entropy_conservation=1-entropy/np.log2(20) if n else np.nan,
                     occupancy_weighted_identity=top/N))
    probability_rows.append(p)
qc = pd.DataFrame(rows)
probabilities = pd.DataFrame(probability_rows, index=np.arange(1, L+1), columns=AA)
qc['gap_caution'] = qc.gap_fraction >= GAP_THRESHOLD
display(qc.round(3).head(12))
print('Gap-rich columns:')
display(qc.loc[qc.gap_caution].round(3))


## 7. Color the alignment two ways
The first view colors residues by broad chemical class (a teaching simplification; histidine is grouped with basic residues). The second colors each occupied cell by **column identity conservation**, using a shared 0–1 scale. Gray cells are gaps; printed letters preserve exact residue identity. The consensus and numeric scores sit directly underneath each block.

**Inspect:** Find a column with chemically similar substitutions and compare it with a column that is strictly identical.


In [ ]:
groups = {'Nonpolar: A V I L M F W P G': ('AVILMFWPG', '#b7d7f0'),
          'Polar: S T N Q': ('STNQ', '#bde5b8'),
          'Acidic: D E': ('DE', '#f4b4a8'),
          'Basic: K R H': ('KRH', '#d1b9e5'),
          'Cysteine: C': ('C', '#ffe38a'), 'Tyrosine: Y': ('Y', '#ffc99a')}
residue_colors = {aa: color for residues, color in groups.values() for aa in residues}

def draw_alignment(mode):
    blocks = (L + BLOCK_WIDTH - 1) // BLOCK_WIDTH
    fig, axes = plt.subplots(blocks, 1, figsize=(18, blocks*(N*.32+2.3)), squeeze=False)
    for block, ax in enumerate(axes[:, 0]):
        start, end = block*BLOCK_WIDTH, min((block+1)*BLOCK_WIDTH, L)
        for i in range(N):
            for j in range(start, end):
                aa = arr[i, j]
                value = qc.iloc[j].identity_conservation
                color = '#dedede' if aa == '-' else (residue_colors[aa] if mode == 'chemistry' else plt.cm.YlGnBu(value))
                ax.add_patch(Rectangle((j-start-.5, i-.5), 1, 1, facecolor=color, edgecolor='white', linewidth=.4))
                ink = 'white' if mode == 'score' and aa != '-' and value > .7 else 'black'
                ax.text(j-start, i, aa, ha='center', va='center', fontsize=10, color=ink, family='monospace')
        for j in range(start, end):
            ax.text(j-start, N+.05, qc.iloc[j].consensus, ha='center', fontsize=10, family='monospace')
            v = qc.iloc[j].identity_conservation
            ax.text(j-start, N+.85, '—' if np.isnan(v) else f'{v:.1f}', ha='center', fontsize=7)
        ax.set_yticks(list(range(N))+[N, N+.8], names+['Consensus', 'Identity'])
        ax.set_xticks(range(end-start), range(start+1, end+1), rotation=90, fontsize=8)
        ax.xaxis.tick_top()
        ax.set_xlim(-.5, end-start-.5)
        ax.set_ylim(N+1.3, -1)
        ax.set_title(f'Alignment columns {start+1}–{end}', pad=30, loc='left')
        ax.tick_params(length=0)
        for spine in ax.spines.values(): spine.set_visible(False)
    fig.suptitle('Residue chemistry' if mode == 'chemistry' else 'Column identity conservation (gaps excluded)', fontsize=18)
    if mode == 'chemistry':
        handles = [Patch(facecolor=c, label=k) for k, (_, c) in groups.items()]
        handles.append(Patch(facecolor='#dedede', label='Gap'))
        fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=10)
    else:
        cax = fig.add_axes([.3, .025, .4, .015])
        fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap='YlGnBu'), cax=cax,
                     orientation='horizontal', label='Most frequent non-gap residue / non-gap count')
    fig.subplots_adjust(top=.90, bottom=.12, left=.16, right=.98, hspace=.8)
    save_figure(fig, 'fig03_alignment_chemistry' if mode == 'chemistry' else 'fig04_alignment_conservation')
draw_alignment('chemistry')
draw_alignment('score')


## 8. Compare conservation and gap profiles
The shared x-axis links all panels to alignment columns. Orange shading flags columns at or above the chosen gap threshold. A high conservation score with low occupancy deserves a different interpretation from a high score supported by every sequence.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True, layout='constrained')
x = qc.column_1_based
axes[0].plot(x, qc.identity_conservation, 'o-', ms=3, label='Identity conservation', color='#1764ab')
axes[0].plot(x, qc.entropy_conservation, '-', label='Entropy conservation', color='#963c87')
axes[0].set_ylabel('Conservation (0–1)')
axes[0].legend(loc='lower left')
axes[1].bar(x, qc.occupancy_weighted_identity, color='#218c74', label='Occupancy-weighted identity')
axes[1].set_ylabel('Weighted identity')
axes[1].legend(loc='lower left')
axes[2].bar(x, qc.gap_fraction, color='#d78523')
axes[2].axhline(GAP_THRESHOLD, color='black', ls='--', label=f'Caution threshold = {GAP_THRESHOLD:.2f}')
axes[2].set_ylabel('Gap fraction')
axes[2].set_xlabel('Alignment column (1-based)')
axes[2].legend(loc='upper right')
for ax in axes:
    ax.set_ylim(-.03, 1.06)
    ax.set_xlim(.5, L+.5)
    ax.grid(axis='y', alpha=.2)
    for col in qc.loc[qc.gap_caution, 'column_1_based']:
        ax.axvspan(col-.5, col+.5, color='#f5b64c', alpha=.13)
axes[0].set_title('Conservation must be interpreted together with coverage')
save_figure(fig, 'fig05_conservation_and_gaps')


## 9. Sequence logo: which residues occur at each position?
This is a **probability logo**: each letter's height is its frequency among non-gap residues. An occupied column sums to 1, whether supported by one sequence or all sequences. An all-gap column is blank. The occupancy track below makes missing coverage visible. Colors use the same chemical groups as the alignment.

Set `LOGO_START` and `LOGO_END` above to zoom into a region; rerun this cell. This logo is **not** a background-corrected information-content logo and has no small-sample correction.


In [ ]:
start, end = int(LOGO_START), min(int(LOGO_END), L)
assert 1 <= start <= end <= L, 'Choose a valid logo window.'
fig, axes = plt.subplots(2, 1, figsize=(max(10, (end-start+1)*.36), 5),
                         sharex=True, gridspec_kw={'height_ratios': [4, 1]}, layout='constrained')
logomaker.Logo(probabilities.loc[start:end], ax=axes[0], color_scheme=residue_colors)
axes[0].set_ylim(0, 1.02)
axes[0].set_ylabel('Non-gap frequency')
axes[0].set_title(f'Cytochrome teaching panel — probability logo, columns {start}–{end}')
axes[1].bar(x, qc.occupancy, color='#566573')
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Occupancy')
axes[1].set_xlim(start-.5, end+.5)
axes[1].set_xticks(range(start, end+1))
axes[1].tick_params(axis='x', labelrotation=90)
axes[1].set_xlabel('Alignment column (1-based)')
save_figure(fig, f'fig06_probability_logo_{start:03d}_{end:03d}')


## 10. Locate CXXCH without mistaking a first cysteine for the motif
Search each **ungapped sequence** for C, two amino acids, C, H, then map all five residues back to alignment columns. Insertions can separate motif residues in the MSA. A consensus match alone would not establish that every sequence contains the motif.

These synthetic sequences were designed to retain a cytochrome-c-like motif. Motif conservation in this panel illustrates a pattern; it does not independently demonstrate biochemical function.


In [ ]:
motif_rows = []
for name in names:
    aligned = aligned_by_id[name]
    mapping = [j+1 for j, aa in enumerate(aligned) if aa != '-']
    ungapped = aligned.replace('-', '')
    for match in re.finditer(r'(?=(C[A-Z]{2}CH))', ungapped):
        cols = mapping[match.start():match.start()+5]
        motif_rows.append(dict(sequence=name, motif=match.group(1),
                               residue_start=match.start()+1,
                               alignment_columns=','.join(map(str, cols))))
motifs = pd.DataFrame(motif_rows, columns=['sequence', 'motif', 'residue_start', 'alignment_columns'])
display(motifs)
missing = sorted(set(names)-set(motifs.sequence))
print('Sequences without CXXCH:', missing or 'None')
print('Fully occupied, strictly conserved columns:')
display(qc.loc[(qc.identity_conservation == 1) & (qc.gap_fraction == 0)])


## 11. Report assignment — answer using your own outputs
Submit your completed notebook and a short report with **at least five numbered figures**: pairwise heatmaps, guide tree, one colored alignment, conservation/gap profiles, and a logo zoomed to the motif. Include both alignment color views if discussing chemical similarity. Every caption must identify the synthetic panel, define colors/axes, cite exact alignment columns where relevant, and explain the observation.

1. **Pairwise evidence (Figure 1):** Calculate N(N−1)/2. Identify the closest pair, including ties, with its identity and distance. Which sequence has the greatest mean distance to the others? Explain one limitation of excluding gaps from identity.
2. **Guide tree (Figure 2):** Locate the closest pair on the tree. Explain how progressive alignment uses a guide tree, why early errors can persist, and why this tree is not a tested evolutionary history. Did this notebook use it to generate the curated MSA?
3. **Alignment colors (Figure 3/4):** Name one strictly conserved column, one variable column, and one gap-containing column, if present. Quote their residues, scores, and non-gap counts. If a category is absent, report that instead. Explain how chemical-class coloring differs from score coloring.
4. **Conservation vs. coverage (Figure 5):** Compare two columns using identity conservation, entropy conservation, and occupancy-weighted identity. Explain how seven identical residues differ from one residue plus six gaps even though both have identity conservation 1.0.
5. **Motif and logo (Figure 6):** Set the logo window to include CXXCH. Give its residue position and mapped alignment columns in one named sequence. Does every sequence retain it? Explain what letter height measures and why occupancy must also be shown.
6. **Evidence and limitations:** Which region provides the strongest evidence of conservation? Support your answer with column numbers and two graphics. Explain why the synthetic design, small sample, sequence relatedness, and alignment uncertainty limit biological inference.
7. **Sensitivity check:** Change GAP_THRESHOLD to 0.50, rerun the scoring and profile cells, and record which columns change caution status. Explain why the underlying scores do not change. Restore your preferred setting and rerun before exporting.

**MICR5203 extension:** Complete the optional MAFFT comparison below and explain whether alignment differences would alter your conclusions.

### Your answers and figure captions
Write your responses here, or in your report. Do not submit plots without interpretations.


## 12. Save tables, alignment, provenance, and a report bundle
The FASTA alignment opens in Jalview or another MSA viewer. CSV/TSV tables preserve unrounded values. The ZIP contains this run's figures, tables, alignment, and provenance; submit your saved notebook separately. Rerunning a figure cell updates that figure within this run.


In [ ]:
pairwise_results.to_csv(OUTPUT_DIR/'pairwise_comparisons.tsv', sep='\t', index=False)
identity.to_csv(OUTPUT_DIR/'identity_matrix.tsv', sep='\t')
distance.to_csv(OUTPUT_DIR/'distance_matrix.tsv', sep='\t')
qc.to_csv(OUTPUT_DIR/'conservation_scores.tsv', sep='\t', index=False)
probabilities.to_csv(OUTPUT_DIR/'residue_probabilities.tsv', sep='\t', index_label='column_1_based')
motifs.to_csv(OUTPUT_DIR/'motif_positions.tsv', sep='\t', index=False)
AlignIO.write(msa, str(OUTPUT_DIR/'jalview_ready_alignment.fasta'), 'fasta')
parameters = dict(notebook_id=NOTEBOOK_ID, version=NOTEBOOK_VERSION,
                  run_id=RUN_ID, sequence_count=N, alignment_columns=L,
                  unique_pairs=expected_pairs, matrix='BLOSUM62',
                  gap_open=GAP_OPEN, gap_extend=GAP_EXTEND,
                  gap_threshold=GAP_THRESHOLD, logo_window=[start, end],
                  msa_status='instructor-curated input; see input hashes',
                  data_scope='original course panel is synthetic; verify provenance of any replacement',
                  reference_id=reference_name, formats=list(FORMATS), png_dpi=300,
                  identity_definition='identical residue pairs / residue-residue pairs; gaps excluded',
                  entropy_definition='1 - H/log2(20); non-gap frequencies; no sequence weighting',
                  input_sources=input_sources,
                  input_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (UNALIGNED_PATH, ALIGNED_PATH)},
                  packages={p: importlib.metadata.version(p) for p in ['biopython','numpy','pandas','matplotlib','logomaker']})
(OUTPUT_DIR/'run_parameters.json').write_text(json.dumps(parameters, indent=2))
assert figure_files and all(p.exists() and p.stat().st_size > 0 for p in figure_files)
pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size} for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]).to_csv(OUTPUT_DIR/'file_manifest.tsv', sep='\t', index=False)
archive = shutil.make_archive(str(OUTPUT_DIR.parent / (RUN_ID + '_report_bundle')), 'zip', OUTPUT_DIR)
print('Report bundle:', archive)
print('PNG figures:')
for p in sorted(OUTPUT_DIR.glob('*.png')): print(' •', p.name)
# Optional Colab download: uncomment the next two lines.
# from google.colab import files
# files.download(archive)


## Optional extension — generate an MSA with MAFFT
The core lesson above uses the curated reference. To generate an alignment yourself in Colab, run the next install cell, then the comparison cell. MAFFT determines its own internal alignment strategy; our displayed tree is not supplied to it. Comparison uses sets of aligned residue pairs, so changed column numbering alone does not count as disagreement. Agreement with a curated teaching alignment is not proof of biological correctness.

After this extension, rerun the export cell to update the ZIP with these additional files.


In [ ]:
# OPTIONAL: uncomment in Colab (Linux) if you are doing the extension.
# !apt-get -qq update
# !apt-get -qq install -y mafft


In [ ]:
RUN_MAFFT = False  # Change to True after installing MAFFT.
if RUN_MAFFT:
    executable = shutil.which('mafft')
    if not executable:
        raise RuntimeError('Install MAFFT first, then rerun this cell.')
    result = subprocess.run([executable, '--auto', str(UNALIGNED_PATH)], capture_output=True, text=True, check=True)
    generated_path = OUTPUT_DIR/'mafft_alignment.fasta'
    generated_path.write_text(result.stdout)
    (OUTPUT_DIR/'mafft_log.txt').write_text(result.stderr)
    generated = AlignIO.read(generated_path, 'fasta')
    generated_dict = {r.id: str(r.seq).upper() for r in generated}
    assert len(generated) == N and set(generated_dict) == set(names)
    assert all(generated_dict[n].replace('-', '') == s for n, s in zip(names, sequences))
    def residue_pairs(aligned):
        positions = {n: 0 for n in names}
        pairs = set()
        for j in range(len(next(iter(aligned.values())))):
            occupied = []
            for name in names:
                if aligned[name][j] != '-':
                    positions[name] += 1
                    occupied.append((name, positions[name]))
            for i, a in enumerate(occupied):
                for b in occupied[i+1:]: pairs.add((a, b))
        return pairs
    reference_pairs, generated_pairs = residue_pairs(aligned_by_id), residue_pairs(generated_dict)
    union = reference_pairs | generated_pairs
    comparison = dict(reference_columns=L, mafft_columns=generated.get_alignment_length(),
                      shared_residue_pairs=len(reference_pairs & generated_pairs),
                      reference_only_pairs=len(reference_pairs-generated_pairs),
                      mafft_only_pairs=len(generated_pairs-reference_pairs),
                      pair_jaccard=len(reference_pairs & generated_pairs)/len(union) if union else None)
    (OUTPUT_DIR/'mafft_comparison.json').write_text(json.dumps(comparison, indent=2))
    display(pd.Series(comparison, name='MAFFT versus curated reference'))
else:
    print('Optional MAFFT extension skipped; core analysis is complete.')


## Methods references
- [Biopython: pairwise alignment](https://biopython.org/docs/latest/Tutorial/chapter_pairwise.html)
- [Logomaker: probability-logo examples](https://logomaker.readthedocs.io/en/latest/examples.html)
- [MAFFT: command-line options](https://mafft.cbrc.jp/alignment/software/manual/manual.html)

**Exit ticket:** A conserved column is most convincing when ______. A sequence logo can conceal missing coverage because ______.
